In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Lambda

In [2]:
corpus = [
    "deep learning models process large scale sequential datasets",
    "neural networks excel in computer vision and natural language processing",
    "continuous bag of words model predicts target words from context"
]

# Tokenization and Vocabulary Building
tokens = " ".join(corpus).lower().split()
vocab = sorted(list(set(tokens)))
vocab_size = len(vocab)
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for i, word in enumerate(vocab)}

print(f"Vocabulary Size: {vocab_size} words")

Vocabulary Size: 27 words


In [3]:
WINDOW_SIZE = 2 # Context window: 2 words left, 2 words right

X, Y = [], []

for sentence in corpus:
    words = sentence.lower().split()
    for i in range(WINDOW_SIZE, len(words) - WINDOW_SIZE):
        # Context words (2 before, 2 after)
        context = [
            word_to_idx[words[i - 2]],
            word_to_idx[words[i - 1]],
            word_to_idx[words[i + 1]],
            word_to_idx[words[i + 2]]
        ]
        target = word_to_idx[words[i]] # Target center word
        X.append(context)
        Y.append(target)

X = np.array(X)
Y = np.array(Y)

In [4]:
EMBEDDING_DIM = 20

cbow_model = Sequential([
    # Embedding layer maps context indices to vector representations
    Embedding(input_dim=vocab_size, output_dim=EMBEDDING_DIM, input_length=WINDOW_SIZE*2),
    # Average context word vectors
    Lambda(lambda x: tf.reduce_mean(x, axis=1)),
    # Softmax output to predict target word probability across vocabulary
    Dense(vocab_size, activation='softmax')
])

cbow_model.compile(optimizer='adam', 
                   loss='sparse_categorical_crossentropy', 
                   metrics=['accuracy'])

cbow_model.fit(X, Y, epochs=200, verbose=0)
print("CBOW Model Training Complete.")


C:\Users\umesh\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



CBOW Model Training Complete.


In [5]:
def predict_target(context_words):
    indices = np.array([[word_to_idx[w] for w in context_words]])
    prediction = cbow_model.predict(indices, verbose=0)
    predicted_idx = np.argmax(prediction[0])
    return idx_to_word[predicted_idx]

# Example test
sample_context = ["continuous", "bag", "words", "model"]
predicted_word = predict_target(sample_context)
print(f"\nContext Words: {sample_context}")
print(f"Predicted Target Word: '{predicted_word}'")


Context Words: ['continuous', 'bag', 'words', 'model']
Predicted Target Word: 'of'
